# 10. DEPLOYMENT — BACKEND
## HairBright · Marketing Mix Modeling — Revenue Prediction

---

**Inputs:**
- `data/models/mmm_trace_B_YYYYMMDD.nc` — posterior trace (nb04)
- `data/processed/hairbright_mmm_features_v2_YYYYMMDD.xlsx` — feature matrix with transformation parameters (nb03)
- `data/outputs/hairbright_attribution_YYYYMMDD.xlsx` — attribution results (nb06)
- `data/outputs/hairbright_optimization_YYYYMMDD.xlsx` — optimization results (nb07)
- `data/outputs/hairbright_validation_YYYYMMDD.xlsx` — validation scorecard (nb08)
- `data/outputs/scorecard_summary.json` — model gate decision (nb08)
- `data/outputs/calibration_factors.json` — iROAS calibration factors (nb08)

**Outputs:**
- `api/` — FastAPI application (`main.py`, `model_backend.py`, `schemas.py`)
- `api/mmm_bundle_YYYYMMDD/` — serialised model bundle (numpy arrays + metadata JSON)
- `api/requirements.txt` — pinned dependencies for the API
- Endpoint tests (inline, pytest-compatible)

**Objective:** Export the fitted Bayesian MMM to a lightweight, serialisable bundle and build
a production-ready FastAPI application exposing five endpoints:

| Endpoint | Method | Description |
|:---------|:-------|:------------|
| `/health` | GET | API liveness check with model gate status |
| `/predict` | POST | Predict revenue from a spend vector |
| `/attribution` | POST | Return channel contribution breakdown |
| `/optimize` | POST | Optimise budget allocation given a total spend |
| `/scenario` | POST | Simulate a what-if spend scenario |

---

> **Pre-requisite:** The model gate loaded from `scorecard_summary.json` (nb08) must show
> **PROCEED** or **PROCEED WITH CAUTION** before deploying. A `caution_flag` is propagated
> to the `/health` endpoint so downstream consumers are always aware of the validation status.


## 10.1. INITIAL SETUP

**Libraries used:**
- `numpy` / `pandas`: Array operations and data handling
- `arviz`: Posterior sample extraction from the `.nc` trace
- `scipy.optimize`: Budget optimiser (same logic as nb07)
- `fastapi` / `uvicorn`: API framework and ASGI server
- `pydantic`: Request/response schema validation
- `pathlib` / `json`: File I/O and bundle serialisation
- `pytest` / `httpx`: In-notebook API tests


In [1]:
# ── Install API dependencies (run once in Colab or fresh environment) ────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  {pkg} installed ✓')
    else:
        print(f'  {pkg} already available ✓')

_ensure('fastapi')
_ensure('uvicorn')
_ensure('httpx')
_ensure('pydantic')
_ensure('openpyxl')
_ensure('arviz')

print('\nAll API dependencies ready.')


  fastapi already available ✓
  uvicorn already available ✓
  httpx already available ✓
  pydantic already available ✓
  openpyxl already available ✓
  arviz already available ✓

All API dependencies ready.


In [2]:
# Imports
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import arviz  as az
from pathlib  import Path
from datetime import datetime
from scipy.optimize import minimize, differential_evolution


In [3]:
# ── Environment detection (Colab vs local) ──────────────────────────────────
IN_COLAB = 'google.colab' in str(globals().get('__builtins__', '')) or \
           os.path.exists('/content')

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PATH_PROJECT = Path(
            '/content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/'
            'marketing-mix-modeling-beauty'
        )
    except Exception as e:
        raise RuntimeError(f'Google Drive mount failed: {e}')
else:
    PATH_PROJECT = Path.cwd().parent   # repo root when running locally

PATH_PROCESSED = PATH_PROJECT / 'data' / 'processed'
PATH_INTERIM   = PATH_PROJECT / 'data' / 'interim'
PATH_MODELS    = PATH_PROJECT / 'data' / 'models'
PATH_OUTPUTS   = PATH_PROJECT / 'data' / 'outputs'
PATH_API       = PATH_PROJECT / 'api'
PATH_API.mkdir(parents=True, exist_ok=True)

print(f'Running in  : {"Colab" if IN_COLAB else "local"} environment')
print(f'Project     : {PATH_PROJECT}')
print(f'API output  : {PATH_API}')


Mounted at /content/drive
Running in  : Colab environment
Project     : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty
API output  : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api


## 10.2. LOAD ALL ARTIFACTS

We load six artifacts produced by previous notebooks:

1. **Feature matrix** (`processed/`) — Hill/adstock parameters, column definitions and scaler stats.
2. **Posterior trace** (`models/`) — MCMC samples for all model parameters.
3. **Attribution results** (`outputs/`) — pre-computed ROAS and contribution summaries.
4. **Validation scorecard** (`outputs/`) — confirms the model is production-ready.
5. **`scorecard_summary.json`** (`outputs/`) — machine-readable gate decision (`PROCEED` or `PROCEED WITH CAUTION`).
6. **`calibration_factors.json`** (`outputs/`) — per-channel iROAS calibration factors from nb08.

> The trace is loaded **read-only** — we extract posterior means and samples, then close it.
> The API bundle does **not** ship the full `.nc` file; instead it serialises the compact
> posterior arrays needed at inference time.
>
> Calibration factors are embedded in the bundle metadata so the API applies them
> consistently in every `/attribution` response without requiring downstream consumers
> to manage them separately.


In [4]:
# ── Column definitions (must match nb04 exactly) ─────────────────────────────
MEDIA_COLS   = ['spend_ps_hill', 'spend_pmax_hill', 'spend_fb_hill', 'spend_ig_hill']
# Control columns must match the v2 production feature matrix from nb03 exactly.
# 'clicks_email_media_adjusted' is the media-adjusted email clicks residual (§3.6b-i).
# 'ig_active' is produced by notebook 03 (§3.8) as a binary Instagram activity
# flag. It is present in the v2 feature matrix Excel sheet but was intentionally
# excluded from the Bayesian model (nb04) due to collinearity with 'spend_ig_hill'
# on active weeks. CONTROL_COLS must therefore contain exactly 6 columns —
# matching the (74, 6) X_ctrl_B array used to fit the posterior trace.
CLICK_COLS   = ['clicks_branded_scaled', 'clicks_organic_scaled', 'clicks_email_media_adjusted']
BINARY_COLS  = ['is_q4', 'is_bf_week', 'is_holiday']
CONTROL_COLS = CLICK_COLS + BINARY_COLS  # 6 controls — ig_active excluded (collinear with spend_ig_hill)
FEATURE_COLS = MEDIA_COLS + CONTROL_COLS
TARGET       = 'log_revenue'

MEDIA_LABELS = {
    'spend_ps_hill'   : 'Paid Search',
    'spend_pmax_hill' : 'Performance Max',
    'spend_fb_hill'   : 'Facebook',
    'spend_ig_hill'   : 'Instagram',
}

# Transformation parameters (must match nb03)
ADSTOCK_PARAMS = {
    'spend_ps'   : 0.2,
    'spend_pmax' : 0.4,
    'spend_fb'   : 0.5,
    'spend_ig'   : 0.5,
}
HILL_ALPHA = {
    'spend_ps'   : 1.5,
    'spend_pmax' : 2.0,
    'spend_fb'   : 2.0,
    'spend_ig'   : 1.5,
}

# Map from Hill-transformed column → raw spend column
SPEND_COL_MAP = {
    'spend_ps_hill'   : 'spend_ps',
    'spend_pmax_hill' : 'spend_pmax',
    'spend_fb_hill'   : 'spend_fb',
    'spend_ig_hill'   : 'spend_ig',
}

print('Column definitions loaded.')
print(f'  Media channels : {MEDIA_COLS}')
print(f'  Controls       : {CONTROL_COLS}')


Column definitions loaded.
  Media channels : ['spend_ps_hill', 'spend_pmax_hill', 'spend_fb_hill', 'spend_ig_hill']
  Controls       : ['clicks_branded_scaled', 'clicks_organic_scaled', 'clicks_email_media_adjusted', 'is_q4', 'is_bf_week', 'is_holiday']


In [5]:
# ── Feature matrix ────────────────────────────────────────────────────────────
feat_files = list(PATH_PROCESSED.glob('hairbright_mmm_features_v2_*.xlsx'))
assert feat_files, 'No feature files found — run notebook 03 first (v2 production matrix).'
FILE_FEAT  = max(feat_files, key=lambda p: p.stat().st_mtime)

df_mmm = pd.read_excel(FILE_FEAT, sheet_name='features', parse_dates=['week'])
df_log = pd.read_excel(FILE_FEAT, sheet_name='transformation_log')

print(f'Feature matrix  : {FILE_FEAT.name}  ({df_mmm.shape[0]} weeks × {df_mmm.shape[1]} cols)')
print(f'Week range       : {df_mmm["week"].min().date()} → {df_mmm["week"].max().date()}')


Feature matrix  : hairbright_mmm_features_v2_20260419.xlsx  (74 weeks × 13 cols)
Week range       : 2022-07-25 → 2023-12-18


In [6]:
# ── Interim dataset — for Hill K computation ──────────────────────────────────
interim_files = list(PATH_INTERIM.glob('hairbright_clean_*.xlsx'))
assert interim_files, 'No interim files — run notebook 01 first.'
FILE_CLEAN = max(interim_files, key=lambda p: p.stat().st_mtime)
df_clean   = pd.read_excel(FILE_CLEAN, parse_dates=['week'])

# Apply spend scale fix (same as nb03)
for col in ['spend_ps', 'spend_pmax', 'spend_fb', 'spend_ig']:
    if col in df_clean.columns:
        mask = df_clean[col] >= 1e6
        df_clean.loc[mask, col] = df_clean.loc[mask, col] / 1e9

# Compute Hill K = median of active-week raw spend
HILL_K = {}
for hill_col, raw_col in SPEND_COL_MAP.items():
    active = df_clean[df_clean[raw_col] > 0][raw_col] if raw_col in df_clean.columns else pd.Series(dtype=float)
    HILL_K[hill_col] = float(active.median()) if len(active) > 0 else 1.0

print(f'Interim dataset  : {FILE_CLEAN.name}')
print('Hill K (saturation half-point):')
for col, k in HILL_K.items():
    print(f'  {col:<25}: ${k:,.2f}')


Interim dataset  : hairbright_clean_20260415.xlsx
Hill K (saturation half-point):
  spend_ps_hill            : $425.14
  spend_pmax_hill          : $2,192.43
  spend_fb_hill            : $1,285.24
  spend_ig_hill            : $434.31


In [7]:
# ── Posterior trace ───────────────────────────────────────────────────────────
trace_files = list(PATH_MODELS.glob('mmm_trace_B_*.nc'))
assert trace_files, 'No trace files found — run notebook 04 first (Model B trace: mmm_trace_B_*.nc).'
FILE_TRACE = max(trace_files, key=lambda p: p.stat().st_mtime)
trace      = az.from_netcdf(str(FILE_TRACE))

n_media = len(MEDIA_COLS)
n_ctrl  = len(CONTROL_COLS)

# Flatten chains × draws
beta_media_samples = trace.posterior['beta_media'].values.reshape(-1, n_media)
beta_ctrl_samples  = trace.posterior['beta_ctrl'].values.reshape(-1, n_ctrl)
intercept_samples  = trace.posterior['intercept'].values.reshape(-1)
n_samples          = beta_media_samples.shape[0]

# Posterior means (used as point estimates in the API)
beta_media_mean = beta_media_samples.mean(axis=0)
beta_ctrl_mean  = beta_ctrl_samples.mean(axis=0)
intercept_mean  = float(intercept_samples.mean())

print(f'Trace file       : {FILE_TRACE.name}')
print(f'Posterior samples: {n_samples:,} (chains × draws)')
print(f'Posterior means  :')
print(f'  intercept      : {intercept_mean:.4f}')
for col, beta in zip(MEDIA_COLS, beta_media_mean):
    print(f'  {col:<25}: {beta:.4f}')


Trace file       : mmm_trace_B_20260419.nc
Posterior samples: 12,000 (chains × draws)
Posterior means  :
  intercept      : 9.5884
  spend_ps_hill            : 0.8781
  spend_pmax_hill          : 1.1437
  spend_fb_hill            : 0.3038
  spend_ig_hill            : 0.3503


In [8]:
# ── Calibration factors (nb08) ────────────────────────────────────────────────
# calibration_factors.json (nb08) contains two relevant sub-dicts:
#
#   "factors"          — {hill_col: float}  ratio = target_iROAS / raw_nb08_iROAS
#   "calibrated_iroas" — {hill_col: {"raw": float, "calibrated": float}}
#
# The factors are derived at the nb08 historical spend level and are NOT
# scale-invariant multipliers. For /attribution we therefore report the
# calibrated iROAS benchmarks directly from "calibrated_iroas" (fixed values
# anchored to industry benchmarks), alongside the model's raw iROAS for the
# requested spend vector.

CAL_FILE = PATH_OUTPUTS / 'calibration_factors.json'

def _parse_cal_json(raw: dict) -> tuple[dict, dict]:
    """
    Returns:
        cal_factors  : {hill_col: float}  — nb08 scaling ratios
        cal_iroas    : {hill_col: float}  — calibrated iROAS benchmark values
    """
    if 'factors' in raw and isinstance(raw['factors'], dict):
        factors = {k: float(v) for k, v in raw['factors'].items()}
    else:
        factors = {k: float(v) for k, v in raw.items() if isinstance(v, (int, float))}

    if 'calibrated_iroas' in raw and isinstance(raw['calibrated_iroas'], dict):
        iroas = {k: float(v['calibrated']) for k, v in raw['calibrated_iroas'].items()}
    else:
        iroas = {}

    return factors, iroas

if CAL_FILE.exists():
    with open(CAL_FILE) as f:
        _cal_raw = json.load(f)
    CALIBRATION_FACTORS, CALIBRATED_IROAS = _parse_cal_json(_cal_raw)
    print('Calibration factors loaded:')
    for ch, val in CALIBRATION_FACTORS.items():
        print(f'  {ch:<25}: factor={val:.5f}  cal_iROAS={CALIBRATED_IROAS.get(ch, "n/a")}x')
else:
    # Fallback — nb08 definitive values
    CALIBRATION_FACTORS = {
        'spend_ps_hill'   : 0.11589,
        'spend_pmax_hill' : 0.31564,
        'spend_fb_hill'   : 0.51817,
        'spend_ig_hill'   : 0.13452,
    }
    CALIBRATED_IROAS = {
        'spend_ps_hill'   : 4.5,
        'spend_pmax_hill' : 3.2,
        'spend_fb_hill'   : 2.8,
        'spend_ig_hill'   : 3.5,
    }
    print('calibration_factors.json not found — using nb08 reference values (fallback).')


Calibration factors loaded:
  spend_ps_hill            : factor=0.11589  cal_iROAS=4.5x
  spend_pmax_hill          : factor=0.31564  cal_iROAS=3.2x
  spend_fb_hill            : factor=0.51817  cal_iROAS=2.8x
  spend_ig_hill            : factor=0.13452  cal_iROAS=3.5x


In [9]:
# ── Validation scorecard — gate check ─────────────────────────────────────────
# The scorecard_summary.json produced by nb08 is the authoritative gate.
# decision: 'PROCEED WITH CAUTION' → caution_flag = True in /health.
# decision: 'PROCEED'              → caution_flag = False.
# decision: 'FAIL'                 → do not deploy; resolve validation issues first.

SCORECARD_JSON = PATH_OUTPUTS / 'scorecard_summary.json'
CAUTION_FLAG   = False
GATE_DECISION  = 'UNKNOWN'
GATE_WARNINGS  = []

if SCORECARD_JSON.exists():
    with open(SCORECARD_JSON) as f:
        scorecard = json.load(f)
    GATE_DECISION = scorecard.get('decision', 'UNKNOWN').upper()
    CAUTION_FLAG  = 'CAUTION' in GATE_DECISION
    print(f'Gate decision    : {GATE_DECISION}')
    print(f'Caution flag     : {CAUTION_FLAG}')
    if CAUTION_FLAG:
        GATE_WARNINGS = scorecard.get('review_items', [])
        print(f'Review items     : {len(GATE_WARNINGS)} (propagated to /health)')
    if 'FAIL' in GATE_DECISION:
        raise RuntimeError(
            'Scorecard decision is FAIL — resolve validation issues before deploying.'
        )
else:
    print('scorecard_summary.json not found — falling back to xlsx scorecard.')
    val_files = list(PATH_OUTPUTS.glob('hairbright_validation_*.xlsx'))
    if val_files:
        FILE_VAL  = max(val_files, key=lambda p: p.stat().st_mtime)
        df_score  = pd.read_excel(FILE_VAL, sheet_name='scorecard')
        n_pass    = (df_score['status'] == 'PASS').sum() if 'status' in df_score.columns else None
        n_total   = len(df_score)
        verdict   = df_score['verdict'].iloc[0] if 'verdict' in df_score.columns else 'UNKNOWN'
        GATE_DECISION = str(verdict).upper()
        CAUTION_FLAG  = 'CAUTION' in GATE_DECISION
        print(f'Validation file  : {FILE_VAL.name}')
        print(f'Scorecard result : {GATE_DECISION}  ({n_pass}/{n_total} checks passed)')
        if 'FAIL' in GATE_DECISION:
            raise RuntimeError('Scorecard FAIL — review nb08 before deploying.')
    else:
        print('WARNING: No validation artifact found. Run notebook 08 first.')
        GATE_DECISION = 'UNKNOWN'
        CAUTION_FLAG  = True


Gate decision    : PROCEED WITH CAUTION
Caution flag     : True
Review items     : 0 (propagated to /health)


## 10.3. MODEL BUNDLE EXPORT

The **model bundle** is a self-contained directory containing:

| File | Contents |
|:-----|:---------|
| `metadata.json` | Column names, transformation parameters, calibration factors, gate decision, model version, export timestamp |
| `beta_media_samples.npy` | Posterior samples for media coefficients `(n_samples, n_media)` |
| `beta_ctrl_samples.npy` | Posterior samples for control coefficients `(n_samples, n_ctrl)` |
| `intercept_samples.npy` | Posterior samples for the intercept `(n_samples,)` |
| `beta_media_mean.npy` | Posterior mean — media |
| `beta_ctrl_mean.npy` | Posterior mean — controls |

**Why not ship the `.nc` trace?** The ArviZ NetCDF trace is typically 50–500 MB and contains
diagnostics, prior predictive, and log-likelihood data not needed at inference time.
The bundle extracts only what the API needs, reducing load time from seconds to milliseconds.

**Posterior subsampling:** the bundle retains at most 2,000 draws — sufficient for 94% HDI
precision while keeping API response times well below 200 ms.

**Calibration factors** and the **gate decision** are embedded in `metadata.json` so the
API is self-contained: no external files need to travel with the bundle at deploy time.


In [10]:
# ── Subsample posterior to 2,000 draws ───────────────────────────────────────
MAX_BUNDLE_SAMPLES = 2_000
rng = np.random.default_rng(42)

if n_samples > MAX_BUNDLE_SAMPLES:
    idx = rng.choice(n_samples, size=MAX_BUNDLE_SAMPLES, replace=False)
    bm_bundle = beta_media_samples[idx]
    bc_bundle = beta_ctrl_samples[idx]
    ic_bundle = intercept_samples[idx]
    print(f'Subsampled {n_samples:,} → {MAX_BUNDLE_SAMPLES:,} posterior draws for bundle.')
else:
    bm_bundle = beta_media_samples
    bc_bundle = beta_ctrl_samples
    ic_bundle = intercept_samples
    print(f'Using all {n_samples:,} posterior draws (below {MAX_BUNDLE_SAMPLES:,} threshold).')


Subsampled 12,000 → 2,000 posterior draws for bundle.


In [11]:
# ── Build and save the bundle ─────────────────────────────────────────────────
process_date = datetime.now().strftime('%Y%m%d')
BUNDLE_DIR   = PATH_API / f'mmm_bundle_{process_date}'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

# ── Control defaults ─────────────────────────────────────────────────────────
ctrl_means    = df_mmm[CONTROL_COLS].mean().values.tolist()
ctrl_defaults = [
    0.0 if col in BINARY_COLS else ctrl_means[i]
    for i, col in enumerate(CONTROL_COLS)
]
ctrl_col_descriptions = {
    col: ('binary flag — 0 at prediction time' if col in BINARY_COLS
          else f'training mean = {ctrl_means[i]:.4f}')
    for i, col in enumerate(CONTROL_COLS)
}

# Metadata JSON — includes gate decision and calibration factors
metadata = {
    'model_name'          : 'HairBright MMM — Bayesian (PyMC)',
    'export_date'         : process_date,
    'trace_file'          : FILE_TRACE.name,
    'n_bundle_samples'    : int(bm_bundle.shape[0]),
    'n_weeks_training'    : int(df_mmm.shape[0]),
    'train_period_start'  : str(df_mmm['week'].min().date()),
    'train_period_end'    : str(df_mmm['week'].max().date()),
    'media_cols'          : MEDIA_COLS,
    'control_cols'        : CONTROL_COLS,
    'feature_cols'        : FEATURE_COLS,
    'target'              : TARGET,
    'media_labels'        : MEDIA_LABELS,
    'adstock_params'      : ADSTOCK_PARAMS,
    'hill_alpha'          : HILL_ALPHA,
    'hill_k'              : {k: float(v) for k, v in HILL_K.items()},
    'spend_col_map'       : SPEND_COL_MAP,
    'intercept_mean'      : float(intercept_mean),
    'beta_media_mean'     : beta_media_mean.tolist(),
    'beta_ctrl_mean'      : beta_ctrl_mean.tolist(),
    'ctrl_defaults'       : ctrl_defaults,
    'ctrl_col_descriptions': ctrl_col_descriptions,
    # Gate and calibration — embedded for self-contained API deployment
    'gate_decision'       : GATE_DECISION,
    'caution_flag'        : CAUTION_FLAG,
    'gate_warnings'       : GATE_WARNINGS,
    'calibration_factors' : CALIBRATION_FACTORS,
    'calibrated_iroas'    : CALIBRATED_IROAS,     # fixed benchmark iROAS per channel (from nb08)
}

with open(BUNDLE_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# NumPy arrays
np.save(BUNDLE_DIR / 'beta_media_samples.npy',  bm_bundle)
np.save(BUNDLE_DIR / 'beta_ctrl_samples.npy',   bc_bundle)
np.save(BUNDLE_DIR / 'intercept_samples.npy',   ic_bundle)
np.save(BUNDLE_DIR / 'beta_media_mean.npy',     beta_media_mean)
np.save(BUNDLE_DIR / 'beta_ctrl_mean.npy',      beta_ctrl_mean)

# Verify bundle
bundle_files = list(BUNDLE_DIR.iterdir())
bundle_size  = sum(f.stat().st_size for f in bundle_files) / 1024 / 1024

print('Bundle saved:')
print(f'  Location   : {BUNDLE_DIR}')
print(f'  Files      : {len(bundle_files)}')
print(f'  Total size : {bundle_size:.2f} MB')
print(f'  Gate       : {GATE_DECISION}  (caution_flag={CAUTION_FLAG})')
for f in sorted(bundle_files):
    print(f'    {f.name:<40} ({f.stat().st_size / 1024:.1f} KB)')


Bundle saved:
  Location   : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api/mmm_bundle_20260420
  Files      : 7
  Total size : 0.19 MB
  Gate       : PROCEED WITH CAUTION  (caution_flag=True)
    beta_ctrl_mean.npy                       (0.2 KB)
    beta_ctrl_samples.npy                    (93.9 KB)
    beta_media_mean.npy                      (0.2 KB)
    beta_media_samples.npy                   (62.6 KB)
    beta_trend_samples.npy                   (15.8 KB)
    intercept_samples.npy                    (15.8 KB)
    metadata.json                            (2.7 KB)


## 10.4. CORE INFERENCE FUNCTIONS

Before writing the API files, we define and test the four core inference functions.
These are the same functions that will be imported inside the FastAPI app, so testing
them here ensures correctness before writing to disk.

**Transformation pipeline (raw spend → Hill-transformed feature):**

```
raw_spend  →  [adstock steady-state]  →  [Hill saturation]  →  beta * x_hill
```

The **adstock steady-state** approximation `x_ss = spend / (1 - decay)` gives the
long-run level for a sustained constant weekly spend — appropriate for predicting
equilibrium revenue effects from a weekly budget, without requiring a full time series.

**Calibrated iROAS** is computed in `/attribution` by applying the channel-specific
calibration factors from nb08. The raw Bayesian iROAS is reported alongside the
calibrated value so the consumer can inspect both.


In [12]:
# ── Transformation helpers ────────────────────────────────────────────────────

def adstock_steady_state(spend: float, decay: float) -> float:
    """Long-run adstock level for a sustained weekly spend."""
    return spend / (1.0 - decay)


def hill_saturation(x: float, alpha: float, K: float) -> float:
    """Hill saturation function: x^alpha / (x^alpha + K^alpha)."""
    if x <= 0:
        return 0.0
    xa = x ** alpha
    Ka = K ** alpha
    return xa / (xa + Ka)


def transform_spend_vector(spend_dict: dict) -> np.ndarray:
    """
    Transform a dict of raw weekly spend values to Hill-transformed features.

    Parameters
    ----------
    spend_dict : dict
        Keys: 'spend_ps', 'spend_pmax', 'spend_fb', 'spend_ig'
        Values: raw weekly spend in USD

    Returns
    -------
    np.ndarray of shape (4,) with Hill-transformed features.
    """
    features = []
    for hill_col, raw_col in SPEND_COL_MAP.items():
        s     = float(spend_dict.get(raw_col, 0.0))
        decay = ADSTOCK_PARAMS[raw_col]
        alpha = HILL_ALPHA[raw_col]
        K     = HILL_K[hill_col]
        x_ss  = adstock_steady_state(s, decay)
        x_h   = hill_saturation(x_ss, alpha, K)
        features.append(x_h)
    return np.array(features, dtype=float)


def default_controls() -> np.ndarray:
    """
    Return control variable vector at 'average non-seasonal' conditions:
    click channels at their training means, seasonal flags off.
    Used when the caller does not provide control values.
    """
    ctrl_means = df_mmm[CONTROL_COLS].mean().values
    result = ctrl_means.copy()
    for i, col in enumerate(CONTROL_COLS):
        if col in BINARY_COLS:
            result[i] = 0.0
    return result


# Quick sanity test
test_spend = {'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000}
test_feats = transform_spend_vector(test_spend)
print('Transform test:')
for col, v in zip(MEDIA_COLS, test_feats):
    print(f'  {col:<25}: {v:.4f}')


Transform test:
  spend_ps_hill            : 0.9826
  spend_pmax_hill          : 0.9737
  spend_fb_hill            : 0.9748
  spend_ig_hill            : 0.9809


In [13]:
# ── /predict — revenue prediction with uncertainty ────────────────────────────

def predict_revenue(
    spend_dict: dict,
    controls: np.ndarray | None = None,
    ci_prob: float = 0.94,
) -> dict:
    """
    Predict weekly revenue for a given spend allocation.

    Returns posterior mean, median, and 94% HDI credible interval of revenue (USD).
    """
    x_media = transform_spend_vector(spend_dict)
    x_ctrl  = controls if controls is not None else default_controls()

    log_rev_samples = (
        ic_bundle
        + bm_bundle @ x_media
        + bc_bundle @ x_ctrl
    )
    rev_samples = np.exp(log_rev_samples)

    alpha     = (1.0 - ci_prob) / 2.0
    ci_lo, ci_hi = np.quantile(rev_samples, [alpha, 1 - alpha])

    return {
        'revenue_mean'    : float(rev_samples.mean()),
        'revenue_median'  : float(np.median(rev_samples)),
        'revenue_ci_lo'   : float(ci_lo),
        'revenue_ci_hi'   : float(ci_hi),
        'ci_prob'         : ci_prob,
        'total_spend'     : float(sum(spend_dict.values())),
        'roas_mean'       : float(rev_samples.mean() / max(sum(spend_dict.values()), 1)),
    }


# Test
pred = predict_revenue(test_spend)
print('Predict test:')
print(f'  Revenue mean    : ${pred["revenue_mean"]:,.0f}')
print(f'  Revenue {pred["ci_prob"]*100:.0f}% HDI : [${pred["revenue_ci_lo"]:,.0f}, ${pred["revenue_ci_hi"]:,.0f}]')
print(f'  ROAS mean       : {pred["roas_mean"]:.2f}x')


Predict test:
  Revenue mean    : $203,991
  Revenue 94% HDI : [$132,792, $296,540]
  ROAS mean       : 10.20x


In [14]:
# ── /attribution — channel contribution breakdown ─────────────────────────────

def attribution(
    spend_dict: dict,
    controls: np.ndarray | None = None,
    calibration: dict | None = None,
    calibrated_iroas: dict | None = None,
) -> dict:
    """
    Return proportional log-linear attribution for each media channel.

    Attribution method: additive share of log(revenue) — each component's
    beta*X term as a fraction of the total predicted mu. Sums to 100% by
    construction.

    iROAS reporting:
      - roas_raw        : model-derived iROAS for the requested spend vector.
      - roas_calibrated : fixed benchmark iROAS from nb08 calibration
                          (anchored to industry priors; spend-invariant).
      - cal_factor      : nb08 scaling ratio (target / raw_nb08) — reported
                          for transparency but not applied to roas_raw here,
                          as it was derived at a specific historical spend level.
    """
    x_media = transform_spend_vector(spend_dict)
    x_ctrl  = controls if controls is not None else default_controls()
    cal     = calibration      if calibration      is not None else CALIBRATION_FACTORS
    cal_iro = calibrated_iroas if calibrated_iroas is not None else CALIBRATED_IROAS

    baseline_samples  = ic_bundle
    media_terms       = bm_bundle * x_media[np.newaxis, :]
    ctrl_term_samples = bc_bundle @ x_ctrl

    mu_samples = baseline_samples + media_terms.sum(axis=1) + ctrl_term_samples

    media_share_samples = media_terms / mu_samples[:, np.newaxis]
    baseline_share      = (baseline_samples / mu_samples).mean()
    ctrl_share          = (ctrl_term_samples / mu_samples).mean()

    channels = {}
    for i, col in enumerate(MEDIA_COLS):
        share_mean    = float(media_share_samples[:, i].mean())
        raw_col       = SPEND_COL_MAP[col]
        spend_usd     = float(spend_dict.get(raw_col, 0.0))
        rev_attr      = float(np.exp(mu_samples).mean() * share_mean)
        roas_raw      = rev_attr / max(spend_usd, 1.0)
        cal_factor    = float(cal.get(col, 1.0))
        roas_cal      = float(cal_iro.get(col, roas_raw))   # benchmark; fallback to raw
        channels[MEDIA_LABELS[col]] = {
            'share_pct'       : round(share_mean * 100, 2),
            'rev_attr_usd'    : round(rev_attr, 2),
            'spend_usd'       : round(spend_usd, 2),
            'roas_raw'        : round(roas_raw, 3),
            'roas_calibrated' : round(roas_cal, 3),
            'cal_factor'      : round(cal_factor, 4),
        }

    return {
        'baseline_share_pct' : round(float(baseline_share) * 100, 2),
        'controls_share_pct' : round(float(ctrl_share) * 100, 2),
        'channels'           : channels,
        'total_spend'        : float(sum(spend_dict.values())),
    }


# Test
attr = attribution(test_spend)
print('Attribution test:')
print(f'  Baseline  : {attr["baseline_share_pct"]}%')
print(f'  Controls  : {attr["controls_share_pct"]}%')
for ch, v in attr['channels'].items():
    print(f'  {ch:<17}: {v["share_pct"]}%  ROAS_raw={v["roas_raw"]}x  ROAS_cal={v["roas_calibrated"]}x  (factor={v["cal_factor"]})')


Attribution test:
  Baseline  : 78.6%
  Controls  : -0.0%
  Paid Search      : 7.03%  ROAS_raw=2.867x  ROAS_cal=4.5x  (factor=0.1159)
  Performance Max  : 9.14%  ROAS_raw=2.331x  ROAS_cal=3.2x  (factor=0.3156)
  Facebook         : 2.42%  ROAS_raw=1.233x  ROAS_cal=2.8x  (factor=0.5182)
  Instagram        : 2.81%  ROAS_raw=1.913x  ROAS_cal=3.5x  (factor=0.1345)


In [15]:
# ── /optimize — budget allocation optimisation ────────────────────────────────

def optimize_budget(
    total_budget: float,
    controls: np.ndarray | None = None,
    method: str = 'SLSQP',
    min_share: float = 0.05,
    max_share: float = 0.60,
    n_posterior_draws: int = 200,
    hdi_prob: float = 0.94,
) -> dict:
    """
    Find the budget allocation across media channels that maximises
    posterior mean predicted revenue subject to:
      - sum(allocations) = total_budget
      - min_share * total_budget <= each channel <= max_share * total_budget

    Parameters
    ----------
    total_budget : float
        Total weekly spend in USD.
    method : str
        'SLSQP' (fast, gradient-based, ~50 ms) or 'DE' (differential evolution,
        global search, ~500 ms).
    min_share : float
        Minimum fraction of total budget per channel (default 5%).
    max_share : float
        Maximum fraction of total budget per channel (default 60%).
    n_posterior_draws : int
        Number of posterior samples used in the objective — fewer is faster.

    Note: At the historical budget of ~$4,382/week the mix is already near the
    mROAS optimum, so reallocations produce only marginal uplift. The meaningful
    opportunity is the +20% budget scenario (+5% revenue).
    """
    if controls is None:
        controls = default_controls()

    rng_opt = np.random.default_rng(0)
    idx     = rng_opt.choice(bm_bundle.shape[0], size=min(n_posterior_draws, bm_bundle.shape[0]), replace=False)
    bm_opt  = bm_bundle[idx]
    ic_opt  = ic_bundle[idx]
    xc_opt  = bc_bundle[idx] @ controls

    raw_cols  = [SPEND_COL_MAP[c] for c in MEDIA_COLS]

    def neg_revenue(alloc):
        spend_d  = dict(zip(raw_cols, alloc))
        x_media  = transform_spend_vector(spend_d)
        log_revs = ic_opt + bm_opt @ x_media + xc_opt
        return -np.exp(log_revs).mean()

    bounds           = [(min_share * total_budget, max_share * total_budget)] * len(MEDIA_COLS)
    slsqp_constraint = [{'type': 'eq', 'fun': lambda a: a.sum() - total_budget}]
    x0               = np.full(len(MEDIA_COLS), total_budget / len(MEDIA_COLS))

    if method.upper() == 'DE':
        from scipy.optimize import NonlinearConstraint
        de_constraint = NonlinearConstraint(lambda a: a.sum(), total_budget, total_budget)
        result = differential_evolution(
            neg_revenue, bounds=bounds, seed=42, maxiter=200, tol=1e-6,
            constraints=de_constraint,
        )
    else:
        result = minimize(neg_revenue, x0, method='SLSQP',
                          bounds=bounds, constraints=slsqp_constraint,
                          options={'ftol': 1e-9, 'maxiter': 500})

    opt_alloc   = dict(zip(raw_cols, result.x))
    pred_opt    = predict_revenue(opt_alloc, controls=controls, ci_prob=hdi_prob)
    pred_equal  = predict_revenue(
        dict(zip(raw_cols, np.full(len(MEDIA_COLS), total_budget / len(MEDIA_COLS)))),
        controls=controls, ci_prob=hdi_prob,
    )

    allocation = {}
    for raw_col, spend in opt_alloc.items():
        hill_col = {v: k for k, v in SPEND_COL_MAP.items()}[raw_col]
        allocation[MEDIA_LABELS[hill_col]] = {
            'spend_usd'  : round(float(spend), 2),
            'share_pct'  : round(float(spend) / total_budget * 100, 2),
        }

    return {
        'total_budget'          : total_budget,
        'method'                : method,
        'converged'             : bool(result.success),
        'optimal_allocation'    : allocation,
        'revenue_optimal_mean'  : pred_opt['revenue_mean'],
        'revenue_optimal_hdi_lo': pred_opt['revenue_ci_lo'],
        'revenue_optimal_hdi_hi': pred_opt['revenue_ci_hi'],
        'roas_optimal'          : pred_opt['roas_mean'],
        'revenue_equal_mean'    : pred_equal['revenue_mean'],
        'uplift_vs_equal_pct'   : round(
            (pred_opt['revenue_mean'] - pred_equal['revenue_mean'])
            / pred_equal['revenue_mean'] * 100, 2
        ),
        'health_status'         : GATE_DECISION,
        'caution_flag'          : CAUTION_FLAG,
    }


# Test (fast SLSQP)
opt = optimize_budget(total_budget=20_000)
print('Optimize test (total = $20,000 / week):')
for ch, v in opt['optimal_allocation'].items():
    print(f'  {ch:<17}: ${v["spend_usd"]:>8,.0f}  ({v["share_pct"]}%)')
print(f'  → Optimal revenue (mean) : ${opt["revenue_optimal_mean"]:,.0f}')
print(f'  → Equal-split revenue    : ${opt["revenue_equal_mean"]:,.0f}')
print(f'  → Uplift vs equal split  : {opt["uplift_vs_equal_pct"]:+.2f}%')
print(f'  → Converged              : {opt["converged"]}')
print(f'  → Health status          : {opt["health_status"]}  (caution={opt["caution_flag"]})')


Optimize test (total = $20,000 / week):
  Paid Search      : $   4,763  (23.82%)
  Performance Max  : $   9,049  (45.24%)
  Facebook         : $   3,672  (18.36%)
  Instagram        : $   2,516  (12.58%)
  → Optimal revenue (mean) : $204,375
  → Equal-split revenue    : $196,397
  → Uplift vs equal split  : +4.06%
  → Converged              : True
  → Health status          : PROCEED WITH CAUTION  (caution=True)


In [16]:
# ── /scenario — what-if scenario simulation ───────────────────────────────────

def scenario(
    scenarios: list[dict],
    controls: np.ndarray | None = None,
    hdi_prob: float = 0.94,
) -> list[dict]:
    """
    Simulate multiple spend scenarios and return predicted revenue for each.

    Parameters
    ----------
    scenarios : list of dicts, each with keys:
        - 'name' : str label for the scenario
        - 'spend_ps', 'spend_pmax', 'spend_fb', 'spend_ig' : float (weekly USD)

    Returns
    -------
    List of scenario results with name, spend, predicted revenue, ROAS and HDI.
    """
    results = []
    for sc in scenarios:
        name      = sc.get('name', 'unnamed')
        spend_d   = {k: sc.get(k, 0.0) for k in ['spend_ps', 'spend_pmax', 'spend_fb', 'spend_ig']}
        pred      = predict_revenue(spend_d, controls=controls, ci_prob=hdi_prob)
        results.append({
            'name'            : name,
            'spend'           : spend_d,
            'total_spend'     : pred['total_spend'],
            'revenue_mean'    : pred['revenue_mean'],
            'revenue_ci_lo'   : pred['revenue_ci_lo'],
            'revenue_ci_hi'   : pred['revenue_ci_hi'],
            'roas_mean'       : pred['roas_mean'],
        })
    return results


# Test
sc_test = [
    {'name': 'Historical mix',  'spend_ps': 512,  'spend_pmax': 2504, 'spend_fb': 1280, 'spend_ig': 85},
    {'name': '+20% budget',     'spend_ps': 614,  'spend_pmax': 3005, 'spend_fb': 1536, 'spend_ig': 102},
    {'name': 'PS-heavy',        'spend_ps': 10000, 'spend_pmax': 3000, 'spend_fb': 3000, 'spend_ig': 4000},
    {'name': 'Social-heavy',    'spend_ps': 3000,  'spend_pmax': 3000, 'spend_fb': 7000, 'spend_ig': 7000},
]
sc_results = scenario(sc_test)

print('Scenario simulation test:')
print(f'{"Scenario":<20} {"Budget":>10} {"Revenue (mean)":>16} {"ROAS":>8}')
print('-' * 58)
for r in sc_results:
    print(f'{r["name"]:<20} ${r["total_spend"]:>9,.0f} ${r["revenue_mean"]:>15,.0f} {r["roas_mean"]:>7.2f}x')


Scenario simulation test:
Scenario                 Budget   Revenue (mean)     ROAS
----------------------------------------------------------
Historical mix       $    4,381 $         87,060   19.87x
+20% budget          $    5,257 $        101,065   19.22x
PS-heavy             $   20,000 $        175,640    8.78x
Social-heavy         $   20,000 $        173,347    8.67x


## 10.5. WRITE API FILES TO DISK

We write three files to `api/`:

| File | Role |
|:-----|:-----|
| `schemas.py` | Pydantic request/response models for all four endpoints |
| `model_backend.py` | Model loader + the four inference functions |
| `main.py` | FastAPI app with routing, CORS, health check and gate propagation |

The bundle directory path is written into `model_backend.py` at generation time.
The `caution_flag` and `gate_decision` from `metadata.json` are surfaced in every
`/health` response and as optional `health_status` fields in inference responses.


In [17]:
# ── schemas.py ────────────────────────────────────────────────────────────────
SCHEMAS_PY = '''
"""Pydantic schemas for the HairBright MMM API."""
from __future__ import annotations
from typing import Optional
from pydantic import BaseModel, Field


# ── Shared spend model ─────────────────────────────────────────────────────────
class SpendInput(BaseModel):
    spend_ps:   float = Field(..., ge=0, description="Paid Search weekly spend (USD)")
    spend_pmax: float = Field(..., ge=0, description="Performance Max weekly spend (USD)")
    spend_fb:   float = Field(..., ge=0, description="Facebook weekly spend (USD)")
    spend_ig:   float = Field(..., ge=0, description="Instagram weekly spend (USD)")


# ── /predict ──────────────────────────────────────────────────────────────────
class PredictRequest(SpendInput):
    hdi_prob: Optional[float] = Field(0.94, ge=0.5, le=0.99, description="HDI probability (default 94%)")

class PredictResponse(BaseModel):
    revenue_mean:   float
    revenue_median: float
    revenue_ci_lo:  float
    revenue_ci_hi:  float
    ci_prob:        float
    total_spend:    float
    roas_mean:      float
    health_status:  str
    caution_flag:   bool
    warnings:       list[str]


# ── /attribution ─────────────────────────────────────────────────────────────
class AttributionRequest(SpendInput):
    pass

class ChannelAttribution(BaseModel):
    share_pct:        float
    rev_attr_usd:     float
    spend_usd:        float
    roas_raw:         float
    roas_calibrated:  float
    cal_factor:       float

class AttributionResponse(BaseModel):
    baseline_share_pct: float
    controls_share_pct: float
    channels:           dict[str, ChannelAttribution]
    total_spend:        float
    health_status:      str
    caution_flag:       bool


# ── /optimize ────────────────────────────────────────────────────────────────
class OptimizeRequest(BaseModel):
    total_budget: float = Field(..., gt=0, description="Total weekly budget in USD")
    min_share:    Optional[float] = Field(0.05, ge=0.0, le=0.5)
    max_share:    Optional[float] = Field(0.60, ge=0.1, le=1.0)
    method:       Optional[str]   = Field("SLSQP", description="SLSQP or DE")
    hdi_prob:     Optional[float] = Field(0.94, ge=0.5, le=0.99)

class ChannelAllocation(BaseModel):
    spend_usd: float
    share_pct: float

class OptimizeResponse(BaseModel):
    total_budget:           float
    method:                 str
    converged:              bool
    optimal_allocation:     dict[str, ChannelAllocation]
    revenue_optimal_mean:   float
    revenue_optimal_hdi_lo: float
    revenue_optimal_hdi_hi: float
    roas_optimal:           float
    revenue_equal_mean:     float
    uplift_vs_equal_pct:    float
    health_status:          str
    caution_flag:           bool


# ── /scenario ────────────────────────────────────────────────────────────────
class ScenarioItem(SpendInput):
    name: str = Field(..., description="Scenario label")

class ScenarioRequest(BaseModel):
    scenarios: list[ScenarioItem]
    hdi_prob:  Optional[float] = Field(0.94, ge=0.5, le=0.99)

class ScenarioResult(BaseModel):
    name:           str
    spend:          dict[str, float]
    total_spend:    float
    revenue_mean:   float
    revenue_ci_lo:  float
    revenue_ci_hi:  float
    roas_mean:      float

class ScenarioResponse(BaseModel):
    results:      list[ScenarioResult]
    health_status: str
    caution_flag:  bool
'''

with open(PATH_API / 'schemas.py', 'w') as f:
    f.write(SCHEMAS_PY.lstrip())
print(f'schemas.py written to {PATH_API / "schemas.py"}')


schemas.py written to /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api/schemas.py


In [18]:
# ── model_backend.py ──────────────────────────────────────────────────────────
_BACKEND_HEADER = (
    f'"""\n'
    f'model_backend.py\n'
    f'HairBright MMM — model loading and inference functions.\n'
    f'\nGenerated by notebook 10 on {process_date}.\n'
    f'Bundle: {BUNDLE_DIR.name}\n'
    f'"""\n'
)

_BACKEND_BUNDLE_PATH = (
    'from __future__ import annotations\n'
    'import json\n'
    'import numpy as np\n'
    'from pathlib import Path\n'
    'from scipy.optimize import minimize, differential_evolution\n\n'
    '# ── Bundle path ───────────────────────────────────────────────────────────────\n'
    '_HERE        = Path(__file__).parent\n'
    + f'BUNDLE_DIR   = _HERE / "{BUNDLE_DIR.name}"\n\n'
)

_BACKEND_BODY = '''
# ── Load bundle ───────────────────────────────────────────────────────────────
with open(BUNDLE_DIR / "metadata.json") as _f:
    META = json.load(_f)

MEDIA_COLS           = META["media_cols"]
CONTROL_COLS         = META["control_cols"]
FEATURE_COLS         = META["feature_cols"]
MEDIA_LABELS         = META["media_labels"]
ADSTOCK              = META["adstock_params"]
HILL_ALPHA           = META["hill_alpha"]
HILL_K               = META["hill_k"]
SPEND_COL_MAP        = META["spend_col_map"]
_cal_raw             = META.get("calibration_factors", {})
CALIBRATION_FACTORS  = (
    {k: float(v) for k, v in _cal_raw["factors"].items()}
    if "factors" in _cal_raw and isinstance(_cal_raw["factors"], dict)
    else {k: float(v) for k, v in _cal_raw.items() if isinstance(v, (int, float))}
)
_iro_raw             = META.get("calibrated_iroas", {})
CALIBRATED_IROAS     = {k: float(v) for k, v in _iro_raw.items()}
GATE_DECISION        = META.get("gate_decision", "UNKNOWN")
CAUTION_FLAG         = META.get("caution_flag", False)
GATE_WARNINGS        = META.get("gate_warnings", [])

bm_bundle = np.load(BUNDLE_DIR / "beta_media_samples.npy")
bc_bundle = np.load(BUNDLE_DIR / "beta_ctrl_samples.npy")
ic_bundle = np.load(BUNDLE_DIR / "intercept_samples.npy")

_CTRL_DEFAULTS = np.array(META.get("ctrl_defaults", [0.0] * len(CONTROL_COLS)), dtype=float)


# ── Transformation helpers ─────────────────────────────────────────────────────
def _hill(x, alpha, K):
    if x <= 0:
        return 0.0
    xa, Ka = x ** alpha, K ** alpha
    return xa / (xa + Ka)


def transform_spend_vector(spend_dict):
    feats = []
    for hill_col, raw_col in SPEND_COL_MAP.items():
        s     = float(spend_dict.get(raw_col, 0.0))
        decay = ADSTOCK[raw_col]
        alpha = HILL_ALPHA[raw_col]
        K     = HILL_K[hill_col]
        x_ss  = s / (1.0 - decay)
        feats.append(_hill(x_ss, alpha, K))
    return np.array(feats, dtype=float)


def _default_controls():
    return _CTRL_DEFAULTS.copy()


# ── /predict ──────────────────────────────────────────────────────────────────
def predict_revenue(spend_dict, controls=None, ci_prob=0.94):
    x_media  = transform_spend_vector(spend_dict)
    x_ctrl   = controls if controls is not None else _default_controls()
    log_revs = ic_bundle + bm_bundle @ x_media + bc_bundle @ x_ctrl
    revs     = np.exp(log_revs)
    alpha    = (1.0 - ci_prob) / 2.0
    lo, hi   = np.quantile(revs, [alpha, 1 - alpha])
    total_sp = float(sum(spend_dict.values()))
    return dict(
        revenue_mean   =float(revs.mean()),
        revenue_median =float(np.median(revs)),
        revenue_ci_lo  =float(lo),
        revenue_ci_hi  =float(hi),
        ci_prob        =ci_prob,
        total_spend    =total_sp,
        roas_mean      =float(revs.mean() / max(total_sp, 1)),
        health_status  =GATE_DECISION,
        caution_flag   =CAUTION_FLAG,
        warnings       =GATE_WARNINGS,
    )


# ── /attribution ─────────────────────────────────────────────────────────────
def attribution(spend_dict, controls=None, calibration=None, calibrated_iroas=None):
    x_media  = transform_spend_vector(spend_dict)
    x_ctrl   = controls if controls is not None else _default_controls()
    cal      = calibration      if calibration      is not None else CALIBRATION_FACTORS
    cal_iro  = calibrated_iroas if calibrated_iroas is not None else CALIBRATED_IROAS
    media_t  = bm_bundle * x_media[np.newaxis, :]
    ctrl_t   = bc_bundle @ x_ctrl
    mu       = ic_bundle + media_t.sum(axis=1) + ctrl_t
    rev_mean = float(np.exp(mu).mean())
    channels = {}
    for i, col in enumerate(MEDIA_COLS):
        share      = float((media_t[:, i] / mu).mean())
        raw_col    = SPEND_COL_MAP[col]
        spend_usd  = float(spend_dict.get(raw_col, 0.0))
        rev_attr   = rev_mean * share
        roas_raw   = rev_attr / max(spend_usd, 1)
        cal_factor = float(cal.get(col, 1.0))
        roas_cal   = float(cal_iro.get(col, roas_raw))  # benchmark; fallback to raw
        channels[MEDIA_LABELS[col]] = dict(
            share_pct       =round(share * 100, 2),
            rev_attr_usd    =round(rev_attr, 2),
            spend_usd       =round(spend_usd, 2),
            roas_raw        =round(roas_raw, 3),
            roas_calibrated =round(roas_cal, 3),
            cal_factor      =round(cal_factor, 4),
        )
    return dict(
        baseline_share_pct =round(float((ic_bundle / mu).mean()) * 100, 2),
        controls_share_pct =round(float((ctrl_t / mu).mean()) * 100, 2),
        channels           =channels,
        total_spend        =float(sum(spend_dict.values())),
        health_status      =GATE_DECISION,
        caution_flag       =CAUTION_FLAG,
    )


# ── /optimize ────────────────────────────────────────────────────────────────
def optimize_budget(
    total_budget,
    controls=None,
    method="SLSQP",
    min_share=0.05,
    max_share=0.60,
    n_draws=200,
    hdi_prob=0.94,
):
    if controls is None:
        controls = _default_controls()
    rng      = np.random.default_rng(0)
    idx      = rng.choice(bm_bundle.shape[0], size=min(n_draws, bm_bundle.shape[0]), replace=False)
    bm_o     = bm_bundle[idx]
    ic_o     = ic_bundle[idx]
    xc_o     = bc_bundle[idx] @ controls
    raw_cols = [SPEND_COL_MAP[c] for c in MEDIA_COLS]

    def neg_rev(alloc):
        x = transform_spend_vector(dict(zip(raw_cols, alloc)))
        return -np.exp(ic_o + bm_o @ x + xc_o).mean()

    bounds     = [(min_share * total_budget, max_share * total_budget)] * len(MEDIA_COLS)
    slsqp_cons = [{"type": "eq", "fun": lambda a: a.sum() - total_budget}]
    x0         = np.full(len(MEDIA_COLS), total_budget / len(MEDIA_COLS))
    if method.upper() == "DE":
        from scipy.optimize import NonlinearConstraint
        de_cons = NonlinearConstraint(lambda a: a.sum(), total_budget, total_budget)
        res = differential_evolution(neg_rev, bounds, seed=42, constraints=de_cons)
    else:
        res = minimize(neg_rev, x0, method="SLSQP", bounds=bounds, constraints=slsqp_cons)

    opt = dict(zip(raw_cols, res.x))
    eq  = dict(zip(raw_cols, np.full(len(MEDIA_COLS), total_budget / len(MEDIA_COLS))))
    p_o = predict_revenue(opt, controls, hdi_prob)
    p_e = predict_revenue(eq,  controls, hdi_prob)

    rev_map   = {v: k for k, v in SPEND_COL_MAP.items()}
    alloc_out = {}
    for rc, s in opt.items():
        hc = rev_map[rc]
        alloc_out[MEDIA_LABELS[hc]] = dict(
            spend_usd=round(float(s), 2),
            share_pct=round(float(s) / total_budget * 100, 2),
        )
    return dict(
        total_budget           =total_budget,
        method                 =method,
        converged              =bool(res.success),
        optimal_allocation     =alloc_out,
        revenue_optimal_mean   =p_o["revenue_mean"],
        revenue_optimal_hdi_lo =p_o["revenue_ci_lo"],
        revenue_optimal_hdi_hi =p_o["revenue_ci_hi"],
        roas_optimal           =p_o["roas_mean"],
        revenue_equal_mean     =p_e["revenue_mean"],
        uplift_vs_equal_pct    =round(
            (p_o["revenue_mean"] - p_e["revenue_mean"]) / p_e["revenue_mean"] * 100, 2
        ),
        health_status          =GATE_DECISION,
        caution_flag           =CAUTION_FLAG,
    )


# ── /scenario ────────────────────────────────────────────────────────────────
def scenario(scenarios, controls=None, hdi_prob=0.94):
    results = []
    for sc in scenarios:
        name = sc.get("name", "unnamed")
        sp   = {k: sc.get(k, 0.0) for k in ["spend_ps", "spend_pmax", "spend_fb", "spend_ig"]}
        p    = predict_revenue(sp, controls, hdi_prob)
        results.append(dict(
            name         =name,
            spend        =sp,
            total_spend  =p["total_spend"],
            revenue_mean =p["revenue_mean"],
            revenue_ci_lo=p["revenue_ci_lo"],
            revenue_ci_hi=p["revenue_ci_hi"],
            roas_mean    =p["roas_mean"],
        ))
    return results
'''

MODEL_BACKEND_PY = _BACKEND_HEADER + _BACKEND_BUNDLE_PATH + _BACKEND_BODY

with open(PATH_API / 'model_backend.py', 'w') as f:
    f.write(MODEL_BACKEND_PY)
print(f'model_backend.py written to {PATH_API / "model_backend.py"}')


model_backend.py written to /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api/model_backend.py


In [19]:
# ── main.py ───────────────────────────────────────────────────────────────────
MAIN_PY = '''
"""
main.py
HairBright MMM — FastAPI application.

Run locally:
    uvicorn main:app --reload --host 0.0.0.0 --port 8000

Endpoints:
    GET  /health           → API liveness + model gate status (caution_flag)
    GET  /model/info       → Model metadata (includes calibration factors)
    POST /predict          → Revenue prediction with 94% HDI
    POST /attribution      → Channel attribution (raw + calibrated iROAS)
    POST /optimize         → Budget allocation optimisation (SLSQP / DE)
    POST /scenario         → What-if scenario simulation
"""
from __future__ import annotations
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from schemas import (
    PredictRequest, PredictResponse,
    AttributionRequest, AttributionResponse,
    OptimizeRequest, OptimizeResponse,
    ScenarioRequest, ScenarioResponse, ScenarioResult,
)
import model_backend as backend

# ── App ───────────────────────────────────────────────────────────────────────
app = FastAPI(
    title       = "HairBright MMM API",
    description = (
        "Marketing Mix Modelling — revenue prediction, attribution, "
        "optimisation and scenario simulation.\n\n"
        "All inference responses include `health_status` and `caution_flag` "
        "reflecting the nb08 validation decision."
    ),
    version     = "1.0.0",
    docs_url    = "/docs",
    redoc_url   = "/redoc",
)

# ── CORS (allow all origins for development; tighten for production) ──────────
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = False,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)


# ── GET /health ───────────────────────────────────────────────────────────────
@app.get("/health", tags=["monitoring"])
def health():
    """
    Liveness check — returns API status, bundle info and model gate decision.

    `caution_flag: true` means the model was deployed with PROCEED WITH CAUTION
    status from nb08 validation. All inference endpoints remain fully operational;
    consumers should display this flag in any UI or dashboard context.
    """
    return {
        "status"        : "ok",
        "model"         : backend.META["model_name"],
        "bundle_date"   : backend.META["export_date"],
        "n_samples"     : int(backend.bm_bundle.shape[0]),
        "health_status" : backend.GATE_DECISION,
        "caution_flag"  : backend.CAUTION_FLAG,
        "warnings"      : backend.GATE_WARNINGS,
    }


# ── GET /model/info ───────────────────────────────────────────────────────────
@app.get("/model/info", tags=["monitoring"])
def model_info():
    """Return full model metadata from the bundle, including calibration factors."""
    return backend.META


# ── POST /predict ─────────────────────────────────────────────────────────────
@app.post("/predict", response_model=PredictResponse, tags=["inference"])
def predict(req: PredictRequest):
    """
    Predict weekly revenue (USD) for a given media spend allocation.

    Returns posterior mean, median, 94% HDI, ROAS and model gate status.
    """
    try:
        spend = {"spend_ps": req.spend_ps, "spend_pmax": req.spend_pmax,
                 "spend_fb": req.spend_fb, "spend_ig":   req.spend_ig}
        return backend.predict_revenue(spend, ci_prob=req.hdi_prob)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))


# ── POST /attribution ─────────────────────────────────────────────────────────
@app.post("/attribution", response_model=AttributionResponse, tags=["inference"])
def attribution_endpoint(req: AttributionRequest):
    """
    Decompose predicted revenue into channel contributions (proportional log-linear).

    Returns baseline, controls and per-channel share (%), attributed revenue (USD),
    raw iROAS, calibrated iROAS and the calibration factor applied.
    """
    try:
        spend = {"spend_ps": req.spend_ps, "spend_pmax": req.spend_pmax,
                 "spend_fb": req.spend_fb, "spend_ig":   req.spend_ig}
        return backend.attribution(spend)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))


# ── POST /optimize ────────────────────────────────────────────────────────────
@app.post("/optimize", response_model=OptimizeResponse, tags=["inference"])
def optimize_endpoint(req: OptimizeRequest):
    """
    Find the budget allocation that maximises predicted revenue
    given a total weekly spend.

    Supports SLSQP (fast, ~50 ms) and DE (global search, ~500 ms).
    At the historical budget of ~$4,382/week the current mix is already
    near the mROAS optimum; the +20% budget scenario (+5% revenue) is the
    recommended path to meaningful uplift.
    """
    try:
        return backend.optimize_budget(
            total_budget =req.total_budget,
            method       =req.method,
            min_share    =req.min_share,
            max_share    =req.max_share,
            hdi_prob     =req.hdi_prob,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))


# ── POST /scenario ────────────────────────────────────────────────────────────
@app.post("/scenario", response_model=ScenarioResponse, tags=["inference"])
def scenario_endpoint(req: ScenarioRequest):
    """
    Simulate multiple spend scenarios and return predicted revenue for each.

    Useful for comparing the historical mix, the SLSQP optimum and the
    +20% budget scenario side-by-side.
    """
    try:
        sc_list = [s.model_dump() for s in req.scenarios]
        results = backend.scenario(sc_list, hdi_prob=req.hdi_prob)
        return ScenarioResponse(
            results      =[ScenarioResult(**r) for r in results],
            health_status=backend.GATE_DECISION,
            caution_flag =backend.CAUTION_FLAG,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))
'''

with open(PATH_API / 'main.py', 'w') as f:
    f.write(MAIN_PY.lstrip())
print(f'main.py written to {PATH_API / "main.py"}')


main.py written to /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api/main.py


In [20]:
# ── requirements.txt ──────────────────────────────────────────────────────────
import importlib.metadata as _imeta

def _ver(pkg):
    try:
        return _imeta.version(pkg)
    except _imeta.PackageNotFoundError:
        return 'unknown'

REQS = (
    f"fastapi=={_ver('fastapi')}\n"
    f"uvicorn[standard]=={_ver('uvicorn')}\n"
    f"pydantic=={_ver('pydantic')}\n"
    f"numpy=={_ver('numpy')}\n"
    f"pandas=={_ver('pandas')}\n"
    f"scipy=={_ver('scipy')}\n"
    f"arviz=={_ver('arviz')}\n"
    "httpx>=0.27\n"
    "openpyxl>=3.1\n"
)

with open(PATH_API / 'requirements.txt', 'w') as f:
    f.write(REQS)

print('requirements.txt written:')
print(REQS)


requirements.txt written:
fastapi==0.135.3
uvicorn[standard]==0.44.0
pydantic==2.12.3
numpy==2.0.2
pandas==2.2.2
scipy==1.16.3
arviz==0.22.0
httpx>=0.27
openpyxl>=3.1



## 10.6. BUNDLE METADATA VERIFICATION

We verify that `metadata.json` is complete and consistent: column definitions,
Hill K values, control defaults, calibration factors and gate decision are all
present before running endpoint tests.


In [21]:
# ── Verify bundle metadata ────────────────────────────────────────────────────
META_PATH = BUNDLE_DIR / 'metadata.json'
with open(META_PATH) as f:
    _meta = json.load(f)

print('Control defaults:')
for col, val in zip(CONTROL_COLS, _meta['ctrl_defaults']):
    print(f'  {col:<35}: {val:.4f}')

print('\nCalibration factors (flat):')
_cal_meta = _meta['calibration_factors']
_cal_flat = (
    _cal_meta['factors'] if 'factors' in _cal_meta and isinstance(_cal_meta['factors'], dict)
    else {k: v for k, v in _cal_meta.items() if isinstance(v, (int, float))}
)
for ch, val in _cal_flat.items():
    print(f'  {ch:<25}: {val:.5f}')

if 'calibrated_iroas' in _meta:
    print('\nCalibrated iROAS benchmarks (nb08):')
    for ch, val in _meta['calibrated_iroas'].items():
        print(f'  {ch:<25}: {val:.2f}x')

print(f'\nGate decision  : {_meta["gate_decision"]}')
print(f'Caution flag   : {_meta["caution_flag"]}')
print(f'Review items   : {len(_meta["gate_warnings"])}')
print(f'\nMetadata keys  : {len(_meta)}')
print('All required keys present ✓' if all(
    k in _meta for k in [
        'gate_decision', 'caution_flag', 'calibration_factors',
        'ctrl_defaults', 'hill_k', 'media_cols',
    ]
) else 'WARNING: missing keys')


Control defaults:
  clicks_branded_scaled              : -0.0000
  clicks_organic_scaled              : -0.0000
  clicks_email_media_adjusted        : 0.0000
  is_q4                              : 0.0000
  is_bf_week                         : 0.0000
  is_holiday                         : 0.0000

Calibration factors (flat):
  spend_ps_hill            : 0.11589
  spend_pmax_hill          : 0.31564
  spend_fb_hill            : 0.51817
  spend_ig_hill            : 0.13452

Calibrated iROAS benchmarks (nb08):
  spend_ps_hill            : 4.50x
  spend_pmax_hill          : 3.20x
  spend_fb_hill            : 2.80x
  spend_ig_hill            : 3.50x

Gate decision  : PROCEED WITH CAUTION
Caution flag   : True
Review items   : 0

Metadata keys  : 26
All required keys present ✓


## 10.7. ENDPOINT TESTS

We test all four endpoints using the inference functions defined in §10.4 — the same
logic that runs inside the FastAPI app. Each test checks:

1. **Schema validity** — response contains expected keys with correct types.
2. **Business logic** — revenue > 0; ROAS > 0; channel shares sum to ≈100%;
   optimised revenue ≥ equal-split revenue; HDI lo < mean < hi.
3. **Gate fields** — `health_status` and `caution_flag` are present and populated.
4. **Calibration** — `/attribution` returns both `roas_raw` and `roas_calibrated`;
   calibrated values are within industry benchmark ranges.
5. **Edge cases** — zero spend, single-channel spend, very large budget.


In [22]:
# ── Test runner ───────────────────────────────────────────────────────────────
import traceback

PASS = '✅'
FAIL = '❌'

def assert_test(name, condition, detail=''):
    status = PASS if condition else FAIL
    print(f'  {status}  {name}' + (f' — {detail}' if detail else ''))
    return condition


all_passed = True

# Calibrated iROAS benchmarks from nb08
CAL_IROAS_RANGES = {
    'Paid Search'      : (2.0, 7.0),
    'Performance Max'  : (1.8, 5.5),
    'Facebook'         : (1.5, 4.5),
    'Instagram'        : (2.0, 6.0),
}

# ── /predict tests ────────────────────────────────────────────────────────────
print('\n── /predict ─────────────────────────────────────────────────────')
try:
    r = predict_revenue({'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000})
    all_passed &= assert_test('revenue_mean > 0',       r['revenue_mean'] > 0)
    all_passed &= assert_test('ROAS > 0',               r['roas_mean'] > 0)
    all_passed &= assert_test('HDI lo < mean < hi',
                               r['revenue_ci_lo'] < r['revenue_mean'] < r['revenue_ci_hi'])
    all_passed &= assert_test('total_spend correct',    abs(r['total_spend'] - 20000) < 1)

    # Edge: zero spend
    r0 = predict_revenue({'spend_ps': 0, 'spend_pmax': 0, 'spend_fb': 0, 'spend_ig': 0})
    all_passed &= assert_test('zero spend → revenue > 0 (baseline only)', r0['revenue_mean'] > 0)

    # Edge: single channel
    r1 = predict_revenue({'spend_ps': 20000, 'spend_pmax': 0, 'spend_fb': 0, 'spend_ig': 0})
    all_passed &= assert_test('single channel → revenue > 0', r1['revenue_mean'] > 0)
except Exception:
    print(f'  {FAIL}  EXCEPTION:', traceback.format_exc())
    all_passed = False


# ── /attribution tests ────────────────────────────────────────────────────────
print('\n── /attribution ─────────────────────────────────────────────────')
try:
    a = attribution({'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000})
    total_share = a['baseline_share_pct'] + a['controls_share_pct'] + \
                  sum(v['share_pct'] for v in a['channels'].values())
    all_passed &= assert_test('shares sum to ≈100%',
                               abs(total_share - 100.0) < 2.0,
                               f'sum={total_share:.2f}%')
    all_passed &= assert_test('all 4 channels present',   len(a['channels']) == 4)
    all_passed &= assert_test('all roas_raw > 0',
                               all(v['roas_raw'] > 0 for v in a['channels'].values()))
    all_passed &= assert_test('all roas_calibrated > 0',
                               all(v['roas_calibrated'] > 0 for v in a['channels'].values()))
    all_passed &= assert_test('baseline_share_pct > 0',   a['baseline_share_pct'] > 0)
    # Calibration sanity: roas_calibrated < roas_raw (factors < 1)
    all_passed &= assert_test('cal_factor is float in (0, 1) for all channels',
                               all(0 < v['cal_factor'] < 1
                                   for v in a['channels'].values()))
    # roas_calibrated = nb08 benchmark (spend-invariant): PS 4.5x / PMax 3.2x / FB 2.8x / IG 3.5x
    all_passed &= assert_test('roas_calibrated in industry benchmark range (1.5x–7x)',
                               all(1.5 <= v['roas_calibrated'] <= 7.0
                                   for v in a['channels'].values()))
except Exception:
    print(f'  {FAIL}  EXCEPTION:', traceback.format_exc())
    all_passed = False


# ── /optimize tests ───────────────────────────────────────────────────────────
print('\n── /optimize ────────────────────────────────────────────────────')
try:
    o = optimize_budget(total_budget=20_000)
    alloc_sum = sum(v['spend_usd'] for v in o['optimal_allocation'].values())
    all_passed &= assert_test('converged',                 o['converged'])
    all_passed &= assert_test('allocation sums to budget',
                               abs(alloc_sum - 20000) < 1.0,
                               f'sum=${alloc_sum:,.2f}')
    all_passed &= assert_test('optimised ≥ equal-split',
                               o['revenue_optimal_mean'] >= o['revenue_equal_mean'] - 1)
    all_passed &= assert_test('all 4 channels allocated', len(o['optimal_allocation']) == 4)
    all_passed &= assert_test('health_status present',    'health_status' in o)
    all_passed &= assert_test('caution_flag present',     'caution_flag' in o)

    # Edge: very large budget (saturation should kick in)
    o_large = optimize_budget(total_budget=500_000)
    all_passed &= assert_test('large budget — converged',  o_large['converged'])
except Exception:
    print(f'  {FAIL}  EXCEPTION:', traceback.format_exc())
    all_passed = False


# ── /scenario tests ───────────────────────────────────────────────────────────
print('\n── /scenario ────────────────────────────────────────────────────')
try:
    sc_list = [
        {'name': 'base',    'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000},
        {'name': 'scale2x', 'spend_ps':10000, 'spend_pmax':16000, 'spend_fb': 8000, 'spend_ig': 6000},
    ]
    sc = scenario(sc_list)
    all_passed &= assert_test('returns 2 results',          len(sc) == 2)
    all_passed &= assert_test('2x budget → higher revenue',
                               sc[1]['revenue_mean'] > sc[0]['revenue_mean'])
    all_passed &= assert_test('2x budget → lower ROAS (diminishing returns)',
                               sc[1]['roas_mean'] <= sc[0]['roas_mean'] + 0.1,
                               f"{sc[0]['roas_mean']:.2f}x → {sc[1]['roas_mean']:.2f}x")
    all_passed &= assert_test('HDI lo < mean for all results',
                               all(r['revenue_ci_lo'] < r['revenue_mean'] for r in sc))
except Exception:
    print(f'  {FAIL}  EXCEPTION:', traceback.format_exc())
    all_passed = False


print()
print('=' * 55)
print(f'TEST SUITE: {"ALL PASSED ✅" if all_passed else "SOME FAILURES ❌"}')
print('=' * 55)



── /predict ─────────────────────────────────────────────────────
  ✅  revenue_mean > 0
  ✅  ROAS > 0
  ✅  HDI lo < mean < hi
  ✅  total_spend correct
  ✅  zero spend → revenue > 0 (baseline only)
  ✅  single channel → revenue > 0

── /attribution ─────────────────────────────────────────────────
  ✅  shares sum to ≈100% — sum=100.00%
  ✅  all 4 channels present
  ✅  all roas_raw > 0
  ✅  all roas_calibrated > 0
  ✅  baseline_share_pct > 0
  ✅  cal_factor is float in (0, 1) for all channels
  ✅  roas_calibrated in industry benchmark range (1.5x–7x)

── /optimize ────────────────────────────────────────────────────
  ✅  converged
  ✅  allocation sums to budget — sum=$20,000.01
  ✅  optimised ≥ equal-split
  ✅  all 4 channels allocated
  ✅  health_status present
  ✅  caution_flag present
  ✅  large budget — converged

── /scenario ────────────────────────────────────────────────────
  ✅  returns 2 results
  ✅  2x budget → higher revenue
  ✅  2x budget → lower ROAS (diminishing returns) 

## 10.8. RESPONSE LATENCY BENCHMARK

We benchmark the inference functions directly (excluding HTTP overhead) to confirm
they meet the <200 ms target for the React dashboard.

| Endpoint | Target | Method |
|:---------|:------:|:------:|
| `/predict` | < 200 ms | Full posterior draw |
| `/attribution` | < 200 ms | Full posterior draw |
| `/optimize` (SLSQP) | < 200 ms | 200-draw subsample |
| `/optimize` (DE) | < 1,000 ms | 200-draw subsample, global |
| `/scenario` (2 items) | < 200 ms | Full posterior draw |


In [23]:
import time

def bench(fn, *args, n=10, label='', **kwargs):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append((time.perf_counter() - t0) * 1000)
    mean_ms = np.mean(times)
    status  = '✅' if mean_ms < 200 else '⚠ SLOW'
    print(f'  {status}  {label:<35}: {mean_ms:6.1f} ms  (n={n})')
    return mean_ms


sp = {'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000}

print('Latency benchmark (bundle size = {:,} samples):'.format(bm_bundle.shape[0]))
bench(predict_revenue,  sp,              label='/predict')
bench(attribution,      sp,              label='/attribution')
bench(optimize_budget,  20_000,          label='/optimize  (SLSQP)')
bench(optimize_budget,  20_000,          label='/optimize  (DE)',     method='DE', n=3)
_bench_sc = [
    {'name': 'base',  'spend_ps': 5000, 'spend_pmax': 8000, 'spend_fb': 4000, 'spend_ig': 3000},
    {'name': '+20%',  'spend_ps': 6000, 'spend_pmax': 9600, 'spend_fb': 4800, 'spend_ig': 3600},
]
bench(scenario,         _bench_sc,       label='/scenario  (2 items)')


Latency benchmark (bundle size = 2,000 samples):
  ✅  /predict                           :    1.0 ms  (n=10)
  ✅  /attribution                       :    0.7 ms  (n=10)
  ✅  /optimize  (SLSQP)                 :   19.5 ms  (n=10)
  ⚠ SLOW  /optimize  (DE)                    : 1069.2 ms  (n=3)
  ✅  /scenario  (2 items)               :    1.5 ms  (n=10)


np.float64(1.5230359997985943)

## 10.9. BUNDLE VERIFICATION

Round-trip test: reload the bundle from disk and verify that every array
matches the in-memory version, and that all metadata fields are intact.


In [24]:
# ── Round-trip bundle verification ───────────────────────────────────────────
with open(BUNDLE_DIR / 'metadata.json') as f:
    meta_disk = json.load(f)

bm_disk = np.load(BUNDLE_DIR / 'beta_media_samples.npy')
bc_disk = np.load(BUNDLE_DIR / 'beta_ctrl_samples.npy')
ic_disk = np.load(BUNDLE_DIR / 'intercept_samples.npy')

assert bm_disk.shape == bm_bundle.shape, f'beta_media shape mismatch: {bm_disk.shape} vs {bm_bundle.shape}'
assert bc_disk.shape == bc_bundle.shape, 'beta_ctrl shape mismatch'
assert ic_disk.shape == ic_bundle.shape, 'intercept shape mismatch'

assert np.allclose(bm_disk, bm_bundle), 'beta_media arrays differ after round-trip'
assert np.allclose(bc_disk, bc_bundle), 'beta_ctrl arrays differ after round-trip'
assert np.allclose(ic_disk, ic_bundle), 'intercept arrays differ after round-trip'

# Verify gate and calibration fields survived serialisation
assert 'gate_decision'      in meta_disk, 'gate_decision missing from metadata'
assert 'calibration_factors' in meta_disk, 'calibration_factors missing from metadata'
assert 'caution_flag'       in meta_disk, 'caution_flag missing from metadata'

print('Bundle round-trip verification:')
print(f'  ✅  metadata.json            — {len(meta_disk)} keys')
print(f'  ✅  beta_media_samples.npy   — shape {bm_disk.shape}')
print(f'  ✅  beta_ctrl_samples.npy    — shape {bc_disk.shape}')
print(f'  ✅  intercept_samples.npy    — shape {ic_disk.shape}')
print(f'  ✅  gate_decision            — {meta_disk["gate_decision"]}')
print(f'  ✅  caution_flag             — {meta_disk["caution_flag"]}')
_cal_disk_raw  = meta_disk['calibration_factors']
_cal_disk_flat = (
    _cal_disk_raw['factors'] if 'factors' in _cal_disk_raw
    else {k: v for k, v in _cal_disk_raw.items() if isinstance(v, (int, float))}
)
print(f'  ✅  calibration_factors      — {list(_cal_disk_flat.keys())}')
print(f'  ✅  All arrays match in-memory versions')

print(f'\nAPI directory ({PATH_API}):')
for p in sorted(PATH_API.rglob('*')):
    if p.is_file():
        size = p.stat().st_size
        size_str = f'{size/1024:.1f} KB' if size < 1e6 else f'{size/1e6:.2f} MB'
        print(f'  {str(p.relative_to(PATH_API)):<50}  {size_str}')


Bundle round-trip verification:
  ✅  metadata.json            — 26 keys
  ✅  beta_media_samples.npy   — shape (2000, 4)
  ✅  beta_ctrl_samples.npy    — shape (2000, 6)
  ✅  intercept_samples.npy    — shape (2000,)
  ✅  gate_decision            — PROCEED WITH CAUTION
  ✅  caution_flag             — True
  ✅  calibration_factors      — ['spend_ps_hill', 'spend_pmax_hill', 'spend_fb_hill', 'spend_ig_hill']
  ✅  All arrays match in-memory versions

API directory (/content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/api):
  main.py                                             6.7 KB
  mmm_bundle_20260413/beta_ctrl_mean.npy              0.2 KB
  mmm_bundle_20260413/beta_ctrl_samples.npy           93.9 KB
  mmm_bundle_20260413/beta_media_mean.npy             0.2 KB
  mmm_bundle_20260413/beta_media_samples.npy          62.6 KB
  mmm_bundle_20260413/intercept_samples.npy           15.8 KB
  mmm_bundle_20260413/metadata.json                   2.3 KB
  mmm_bundle_20260420/be

## 10.10. DEPLOYMENT INSTRUCTIONS

### Local development

```bash
cd api/
pip install -r requirements.txt
uvicorn main:app --reload --host 0.0.0.0 --port 8000
```

Interactive docs: [http://localhost:8000/docs](http://localhost:8000/docs)

### Quick smoke test (curl)

```bash
# Health check — inspect caution_flag before using the API
curl http://localhost:8000/health

# Predict
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"spend_ps":512,"spend_pmax":2504,"spend_fb":1280,"spend_ig":85}'

# Attribution (returns raw + calibrated iROAS)
curl -X POST http://localhost:8000/attribution \
  -H 'Content-Type: application/json' \
  -d '{"spend_ps":512,"spend_pmax":2504,"spend_fb":1280,"spend_ig":85}'

# Optimize
curl -X POST http://localhost:8000/optimize \
  -H 'Content-Type: application/json' \
  -d '{"total_budget":4382,"method":"SLSQP"}'

# Scenario — historical mix vs +20% budget
curl -X POST http://localhost:8000/scenario \
  -H 'Content-Type: application/json' \
  -d '{"scenarios":[
    {"name":"historical","spend_ps":512,"spend_pmax":2504,"spend_fb":1280,"spend_ig":85},
    {"name":"+20% budget","spend_ps":614,"spend_pmax":3005,"spend_fb":1536,"spend_ig":102}
  ]}'
```

### Cloud deployment (Google Cloud Run)

1. Add a `Dockerfile` to `api/`:

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
EXPOSE 8080
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
```

2. Build and push:

```bash
gcloud builds submit --tag gcr.io/YOUR_PROJECT/hairbright-mmm-api
gcloud run deploy hairbright-mmm-api \
    --image gcr.io/YOUR_PROJECT/hairbright-mmm-api \
    --platform managed \
    --region europe-west1 \
    --allow-unauthenticated
```

3. The deployed URL is used as `VITE_API_BASE_URL` in the React dashboard (notebook 11).

### Monthly refresh procedure

When new data becomes available, follow these steps in order:

1. Re-run notebooks 01–03 with the extended dataset to produce an updated feature matrix.
2. Re-run notebook 04 to fit a new trace (`mmm_trace_B_YYYYMMDD.nc`).
3. Re-run notebooks 05–07 for diagnostics, attribution and optimisation.
4. Re-run notebook 08 to produce an updated scorecard and calibration factors.
5. Re-run **this notebook** to export a new bundle (`mmm_bundle_YYYYMMDD/`) and regenerate the API files.
6. Redeploy the API.

No schema changes are expected unless media channels are added or renamed.

> **Note:** The `caution_flag` in `/health` reflects the new scorecard decision automatically — no code change required.

## 10.11. SUMMARY

In [25]:
print('=' * 65)
print('DEPLOYMENT BACKEND SUMMARY')
print('=' * 65)

print(f'\nModel            : {meta_disk["model_name"]}')
print(f'Trace file       : {meta_disk["trace_file"]}')
print(f'Training period  : {meta_disk["train_period_start"]} → {meta_disk["train_period_end"]}')
print(f'Weeks trained    : {meta_disk["n_weeks_training"]}')
print(f'Bundle samples   : {meta_disk["n_bundle_samples"]:,}')
print(f'Bundle location  : {BUNDLE_DIR.relative_to(PATH_PROJECT)}')
print(f'Gate decision    : {meta_disk["gate_decision"]}  (caution_flag={meta_disk["caution_flag"]})')

print('\n--- Calibration factors (nb08) ---')
for ch, val in meta_disk['calibration_factors'].items():
    print(f'  {ch:<12}: {val}')

print('\n--- API files ---')
for fname in ['main.py', 'model_backend.py', 'schemas.py', 'requirements.txt']:
    fpath = PATH_API / fname
    size  = fpath.stat().st_size if fpath.exists() else 0
    print(f'  {fname:<25}: {size / 1024:.1f} KB')

print('\n--- Endpoints ---')
for method, path, desc in [
    ('GET',  '/health',      'Liveness + gate status (caution_flag)'),
    ('GET',  '/model/info',  'Full metadata (incl. calibration factors)'),
    ('POST', '/predict',     'Revenue prediction with 94% HDI'),
    ('POST', '/attribution', 'Attribution — raw + calibrated iROAS'),
    ('POST', '/optimize',    'Budget allocation (SLSQP / DE)'),
    ('POST', '/scenario',    'What-if scenario simulation'),
]:
    print(f'  {method:<5} {path:<18}: {desc}')

print('\n--- Test suite ---')
print(f'  Result: {"ALL PASSED ✅" if all_passed else "SOME FAILURES ❌"}')

print('\n--- Next step ---')
print('  → Notebook 11: pilot_react.ipynb')
print('    Connect the React dashboard to this API.')
print('    Highlight the +20% budget scenario (+5% revenue) as the primary recommendation.')
print('    Use /health caution_flag to show the PROCEED WITH CAUTION badge in the UI.')


DEPLOYMENT BACKEND SUMMARY

Model            : HairBright MMM — Bayesian (PyMC)
Trace file       : mmm_trace_B_20260419.nc
Training period  : 2022-07-25 → 2023-12-18
Weeks trained    : 74
Bundle samples   : 2,000
Bundle location  : api/mmm_bundle_20260420
Gate decision    : PROCEED WITH CAUTION  (caution_flag=True)

--- Calibration factors (nb08) ---
  spend_ps_hill: 0.11589
  spend_pmax_hill: 0.31564
  spend_fb_hill: 0.518166
  spend_ig_hill: 0.134522

--- API files ---
  main.py                  : 6.7 KB
  model_backend.py         : 9.1 KB
  schemas.py               : 3.9 KB
  requirements.txt         : 0.1 KB

--- Endpoints ---
  GET   /health           : Liveness + gate status (caution_flag)
  GET   /model/info       : Full metadata (incl. calibration factors)
  POST  /predict          : Revenue prediction with 94% HDI
  POST  /attribution      : Attribution — raw + calibrated iROAS
  POST  /optimize         : Budget allocation (SLSQP / DE)
  POST  /scenario         : What-if scena